# 大类资产配置策略落地方法研究

## 国泰君安量化配置团队

### 报告日期：2024年1月11日

---


## 一、研究背景与目的

本报告旨在探讨大类资产配置策略的落地方法，通过量化模型实现资产配置的系统化和可执行性。我们测试了以下经典资产配置模型：

1. **Black-Litterman模型 (BL)** - 结合观点的贝叶斯框架
2. **风险平价模型 (Risk Parity)** - 风险均衡配置策略
3. **宏观因子模型 (Macro Factor)** - 基于宏观因子的配置
4. **最小方差模型 (Min Variance)** - 组合方差最小化

### 资产配置池

- 沪深300 - A股大盘
- 上证50 - A股蓝筹
- 新华商品指数 - 商品
- 债券指数 - 利率债
- 标普500 - 美股大盘
- 纳斯达克100 - 美股科技

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

## 二、数据概况

### 2.1 数据来源

补充数据文件 `data/asset_alloc_Data.csv`，包含多资产历史价格数据。

### 2.2 回测时间区间

- 起始日期：2021年1月1日
- 结束日期：2024年1月11日
- 交易日数量：710天

---


In [ ]:
results_df = pd.read_csv('../output/backtest_results_multi.csv', encoding='utf-8-sig')
print("回测结果数据：")
results_df

## 三、回测结果分析

### 3.1 各模型表现汇总


In [ ]:
display_df = results_df.copy()
display_df['年化收益'] = display_df['annualized_return'].apply(lambda x: f'{x:.2%}')
display_df['年化波动率'] = display_df['volatility'].apply(lambda x: f'{x:.2%}')
display_df['夏普比率'] = display_df['sharpe_ratio'].apply(lambda x: f'{x:.2f}')
display_df['最大回撤'] = display_df['max_drawdown'].apply(lambda x: f'{x:.2%}')
display_df['胜率'] = display_df['win_ratio'].apply(lambda x: f'{x:.2%}')

summary = display_df[['model', '年化收益', '年化波动率', '夏普比率', '最大回撤', '胜率']].rename(columns={'model': '模型'})
print("=" * 70)
print("                        回测结果汇总表")
print("=" * 70)
summary

### 3.2 净值曲线对比


In [ ]:
from IPython.display import Image, display
display(Image('../output/equity_curve_multi.png', width=900))

### 3.3 关键指标分析

| 模型 | 年化收益 | 年化波动率 | 夏普比率 | 最大回撤 |
|------|----------|------------|----------|----------|
| BL模型1 | -1.11% | 10.24% | -0.11 | 23.30% |
| BL模型2 | -1.21% | 10.47% | -0.12 | 20.60% |
| 风险平价模型 | 2.25% | 9.01% | 0.25 | 14.92% |
| 宏观因子模型 | -1.33% | 10.21% | -0.13 | 23.72% |
| 最小方差模型 | 2.46% | 8.09% | 0.30 | 15.07% |

**注**：权重上限约束为30%，避免单一资产过度集中。

---


## 四、资产权重分析

### 4.1 期末配置权重

以下为各模型在回测期末（2024年1月）的资产权重配置：


In [ ]:
weights_data = {
    '沪深300': [0.05, 0.05, 0.1556, 0.05, 0.0921],
    '新华商品指数': [0.05, 0.05, 0.2410, 0.05, 0.3000],
    '债券指数': [0.25, 0.25, 0.1397, 0.25, 0.3000],
    '标普500': [0.30, 0.30, 0.1781, 0.30, 0.1703],
    '上证50': [0.05, 0.05, 0.1561, 0.05, 0.0876],
    '纳斯达克100': [0.30, 0.30, 0.1296, 0.30, 0.0500]
}

weights_df = pd.DataFrame(weights_data, index=['BL模型1', 'BL模型2', '风险平价模型', '宏观因子模型', '最小方差模型']).T
print("期末资产配置权重：")
print("-" * 70)
print((weights_df * 100).round(1).to_string() + "%")

### 4.2 权重配置特点

1. **风险平价模型**：债券权重较高(30%)，商品(24%)和美股(17%)作为补充，实现风险均衡
2. **最小方差模型**：债券和商品各30%，追求组合方差最小化
3. **BL模型/宏观因子模型**：受权重上限约束，美股(30%)、债券(25%)占主导，A股权重受限

---


## 五、结果讨论与分析

### 5.1 模型表现评估

1. **最小方差模型表现最佳**：年化收益2.46%，夏普比率0.30，波动率最低(8.09%)

2. **风险平价模型次之**：年化收益2.25%，夏普比率0.25，较好地实现了风险分散

3. **BL模型和宏观因子模型表现一般**：
   - 主要原因：2021-2024年期间A股(沪深300、上证50)表现较差(-18.48%、-14.88%年化收益)
   - 美股(纳斯达克100、标普500)表现较好(18.99%、13.39%年化收益)
   - BL模型基于历史观点，在极端市场环境下难以准确预测

### 5.2 市场环境分析

2021-2024年市场特征：
- A股市场：沪深300年化收益-18.48%，整体熊市
- 美股市场：纳斯达克100年化收益18.99%，科技股强势
- 商品市场：新华商品指数年化收益11.02%
- 债券市场：债券指数年化收益4.84%，表现稳健

### 5.3 配置建议

1. **风险平价和最小方差模型**：在当前市场环境下表现稳健，适合稳健型投资者

2. **BL模型**：需要更准确的观点输入才能发挥优势，当前简化版本表现受限

3. **权重约束**：30%上限有效控制了单一资产风险暴露，避免了纳斯达克100的过度集中

---


## 六、结论

本研究使用多资产配置框架，对五种量化配置模型进行了回测验证。主要结论：

1. **风险平价和最小方差模型**在2021-2024年市场环境下表现最优，实现了正收益和较好的风险调整后收益

2. **BL模型和宏观因子模型**受A股市场拖累，表现一般，但在美股占主导的市场环境中仍保持了相对稳健的回撤控制

3. **权重约束（30%上限）**有效避免了单一资产风险过度集中，提高了组合的稳健性

4. **多资产配置**相比单一资产配置，确实能够有效分散风险，降低组合波动率

---

**数据说明**：本报告使用补充数据文件 `data/asset_alloc_Data.csv`，包含沪深300、上证50、新华商品指数、债券指数、标普500、纳斯达克100等资产的历史价格数据。回测区间为2021年1月至2024年1月。